# NB03. Surrogate Zoo: Prediction & Attribution Fidelity

**목적**: 8개 대리 방법을 비교하여 (1) R²와 AdvTop-k의 독립성(C1)을 실증하고, (2) depth-1 = TreeSHAP 동치(C3)를 검증하며, (3) Ridge의 구조적 실패를 분석한다.

**Dependencies**: NB01 (datasets.pkl), NB02 (bb_shap, bb_score, train_val_idx)

**Outputs** (`outputs/NB03/`):
- `results.json`: 방법별 {R², Spearman, Top-k, AdvTop-k, AdvFull}
- `contribs_{method}_{dataset}.npy`: 방법별 contributions (downstream 재사용)

**8개 방법**:
1. Tree d1 (제안), 2. Tree d1+mono, 3. Tree d3, 4. Tree d6
5. EBM (GAM), 6. EBM+mono, 7. Linear Ridge, 8. OptBin+Ridge

**핵심 질문**:
- R²가 높아도 AdvTop-k가 낮을 수 있는가? (C1)
- Tree d1의 pred_contrib는 TreeSHAP과 동치인가? (C3)
- Ridge의 AdvTop-k 실패는 어떤 구조적 원인인가? (C3)

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pickle, json, time
import numpy as np
import pandas as pd
import shap
from pathlib import Path
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

from decentra.surrogate import TreeSurrogate, EBMSurrogate, LinearSurrogate, BinningSurrogate
from decentra.metrics.attribution import topk, advtopk, advfull, random_baseline_advtopk

NB01_DIR = Path('../outputs/NB01')
NB02_DIR = Path('../outputs/NB02')
OUT_DIR = Path('../outputs/NB03')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load data
with open(NB01_DIR / 'datasets.pkl', 'rb') as f:
    datasets = pickle.load(f)

# Load BB artifacts
bb = {}
for name in datasets:
    bb[name] = {
        'shap': np.load(NB02_DIR / f'bb_shap_{name}.npy'),
        'score_test': np.load(NB02_DIR / f'bb_score_test_{name}.npy'),
        'score_train': np.load(NB02_DIR / f'bb_score_train_{name}.npy'),
        'prob': np.load(NB02_DIR / f'bb_prob_{name}.npy'),
    }
    with open(NB02_DIR / f'train_val_idx_{name}.pkl', 'rb') as f:
        bb[name]['tv_idx'] = pickle.load(f)
    print(f'{name}: shap={bb[name]["shap"].shape}, score range=[{bb[name]["score_test"].min()}, {bb[name]["score_test"].max()}]')

GMSC: shap=(30000, 10), score range=[394, 818]
HC: shap=(61503, 41), score range=[465, 800]


## 1. Define Surrogates & Evaluation

In [2]:
def evaluate(surr_contribs, surr_pred, bb_shap, bb_score, bb_prob, n_features):
    """Compute all prediction + attribution fidelity metrics."""
    from decentra.metrics.attribution import attribution_fidelity, random_baseline_advtopk
    
    r2 = round(r2_score(bb_score, surr_pred), 4)
    sp = round(float(spearmanr(bb_score, surr_pred)[0]), 4)
    reject = bb_prob >= np.percentile(bb_prob, 90)
    af = attribution_fidelity(bb_shap, surr_contribs, reject, bb_sign=1, surr_sign=-1)
    rb1 = round(random_baseline_advtopk(bb_shap, reject, 1, n_features, bb_sign=1), 4)
    rb4 = round(random_baseline_advtopk(bb_shap, reject, 4, n_features, bb_sign=1), 4)
    
    return {'R2': r2, 'Spearman': sp,
            **{k: round(v, 4) for k, v in af.items()},
            'Random_AT1': rb1, 'Random_AT4': rb4}

N_JOBS_EBM = 4  # 28 cores available

SURROGATES = {
    'Tree-d1':       lambda: TreeSurrogate(max_depth=1, verbose=0),
    'Tree-d1+mono':  lambda: TreeSurrogate(max_depth=1, monotone_detect_mode='auto', verbose=0),
    'Tree-d3':       lambda: TreeSurrogate(max_depth=3, verbose=0),
    'Tree-d6':       lambda: TreeSurrogate(max_depth=6, verbose=0),
    'EBM':           lambda: EBMSurrogate(interactions=0, n_jobs=N_JOBS_EBM),
    'EBM+mono':      lambda: EBMSurrogate(interactions=0, monotone_detect_mode='auto', n_jobs=N_JOBS_EBM),
    'Ridge':         lambda: LinearSurrogate(method='ridgecv'),
    'OptBin+Ridge':  lambda: BinningSurrogate(max_n_bins=10),
}
print(f'{len(SURROGATES)} methods, EBM n_jobs={N_JOBS_EBM}')

8 methods, EBM n_jobs=4


## 2. Train & Evaluate All Surrogates

In [3]:
all_results = {}

for data_name in datasets:
    d = datasets[data_name]
    X_train, X_test = d['X_train'], d['X_test']
    feature_names = d['feature_names']
    y_score_train = bb[data_name]['score_train']
    y_score_test = bb[data_name]['score_test']
    bb_shap_test = bb[data_name]['shap']
    bb_prob_test = bb[data_name]['prob']
    n_features = len(feature_names)
    
    # Val set for early stopping
    tv = bb[data_name]['tv_idx']
    X_tr = X_train.loc[tv['train_idx']]
    X_val = X_train.loc[tv['val_idx']]
    y_tr = y_score_train[X_train.index.get_indexer(tv['train_idx'])]
    y_val = y_score_train[X_train.index.get_indexer(tv['val_idx'])]
    
    print(f'\n{"="*70}')
    print(f'  {data_name}: {n_features} features, {len(X_test):,} test')
    print(f'{"="*70}')
    
    all_results[data_name] = {}
    
    for method_name, make_surr in SURROGATES.items():
        t0 = time.time()
        surr = make_surr()
        
        if isinstance(surr, TreeSurrogate):
            surr.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        elif isinstance(surr, BinningSurrogate):
            surr.fit(X=X_train, y_logit=y_score_train, feature_names=feature_names)
        else:
            surr.fit(X=X_train, y_logit=y_score_train)
        
        surr_pred = surr.predict(X_test)
        surr_contribs = surr.contributions(X_test)
        
        metrics = evaluate(surr_contribs, surr_pred,
                          bb_shap_test, y_score_test, bb_prob_test, n_features)
        metrics['time_s'] = round(time.time() - t0, 1)
        
        all_results[data_name][method_name] = metrics
        np.save(OUT_DIR / f'contribs_{method_name}_{data_name}.npy',
                surr_contribs.astype(np.float32))
        
        print(f'  {method_name:16s}  R2={metrics["R2"]:.3f}  '
              f'AT1={metrics["AdvTop1"]:.3f}  AT4={metrics["AdvTop4"]:.3f}  '
              f'AFR={metrics["AdvFull_R"]:.3f}  ({metrics["time_s"]}s)')
    
    rb = list(all_results[data_name].values())[0]
    print(f'  {"Random":<16s}  {"":>10s}  '
          f'AT1={rb["Random_AT1"]:.3f}  AT4={rb["Random_AT4"]:.3f}')


  GMSC: 10 features, 30,000 test


  Tree-d1           R2=0.936  AT1=0.808  AT4=0.930  AFR=0.903  (0.8s)


  Tree-d1+mono      R2=0.927  AT1=0.803  AT4=0.916  AFR=0.909  (1.9s)


  Tree-d3           R2=0.979  AT1=0.906  AT4=0.951  AFR=0.967  (4.4s)


  Tree-d6           R2=0.990  AT1=0.955  AT4=0.966  AFR=0.964  (25.3s)


...

  Tree-d6           R2=0.972  AT1=0.975  AT4=0.911  AFR=0.913  (62.9s)


  EBM               R2=0.926  AT1=0.937  AT4=0.844  AFR=0.885  (397.9s)


  EBM+mono          R2=0.834  AT1=0.926  AT4=0.546  AFR=0.709  (66.6s)


  Ridge             R2=0.829  AT1=0.000  AT4=0.616  AFR=0.797  (3.5s)


  OptBin+Ridge      R2=0.863  AT1=0.911  AT4=0.622  AFR=0.785  (7.0s)
  Random                        AT1=0.024  AT4=0.098


## 3. Depth-1 Equivalence Check (C3)

Tree d1의 `pred_contrib`(= `contributions()`)이 `shap.TreeExplainer`와 수학적으로 동치인지 검증한다.

In [4]:
for data_name in datasets:
    d = datasets[data_name]
    X_test = d['X_test']
    tv = bb[data_name]['tv_idx']
    X_tr = d['X_train'].loc[tv['train_idx']]
    X_val = d['X_train'].loc[tv['val_idx']]
    y_tr = bb[data_name]['score_train'][d['X_train'].index.get_indexer(tv['train_idx'])]
    y_val = bb[data_name]['score_train'][d['X_train'].index.get_indexer(tv['val_idx'])]
    
    surr = TreeSurrogate(max_depth=1, verbose=0)
    surr.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    
    # pred_contrib (decentra)
    contrib_pred = surr.contributions(X_test.iloc[:200])
    
    # shap.TreeExplainer (independent)
    shap_vals = np.array(shap.TreeExplainer(surr.model_).shap_values(X_test.iloc[:200]))
    
    max_diff = np.max(np.abs(contrib_pred - shap_vals))
    is_equal = np.allclose(contrib_pred, shap_vals, atol=1e-6)
    
    print(f'{data_name}: max |pred_contrib - TreeSHAP| = {max_diff:.2e}  →  '
          f'{"EQUIVALENT" if is_equal else "DIFFERENT"}')

GMSC: max |pred_contrib - TreeSHAP| = 0.00e+00  →  EQUIVALENT


HC: max |pred_contrib - TreeSHAP| = 0.00e+00  →  EQUIVALENT


## 4. Ridge Structural Failure Analysis (C3)

BB SHAP feature importance vs Ridge coefficients vs Tree-d1 SHAP. Ridge가 다중공선성으로 기여를 재분배하는 구조를 시각화한다.

In [5]:
for data_name in datasets:
    feature_names = datasets[data_name]['feature_names']
    bb_imp = np.abs(bb[data_name]['shap']).mean(axis=0)
    
    ridge_contribs = np.load(OUT_DIR / f'contribs_Ridge_{data_name}.npy')
    tree_contribs = np.load(OUT_DIR / f'contribs_Tree-d1_{data_name}.npy')
    
    ridge_imp = np.abs(ridge_contribs).mean(axis=0)
    tree_imp = np.abs(tree_contribs).mean(axis=0)
    
    df = pd.DataFrame({
        'Feature': feature_names,
        'BB_SHAP': bb_imp,
        'Ridge': ridge_imp,
        'Tree_d1': tree_imp,
    })
    
    # Rank comparison
    df['Rank_BB'] = df['BB_SHAP'].rank(ascending=False).astype(int)
    df['Rank_Ridge'] = df['Ridge'].rank(ascending=False).astype(int)
    df['Rank_Tree'] = df['Tree_d1'].rank(ascending=False).astype(int)
    df = df.sort_values('Rank_BB')
    
    print(f'\n=== {data_name}: Feature Importance Comparison (Top 10) ===')
    print(df[['Feature', 'Rank_BB', 'BB_SHAP', 'Rank_Ridge', 'Ridge', 'Rank_Tree', 'Tree_d1']]
          .head(10).to_string(index=False))
    
    # Spearman between importance rankings
    rho_ridge = spearmanr(bb_imp, ridge_imp)[0]
    rho_tree = spearmanr(bb_imp, tree_imp)[0]
    print(f'\n  Spearman(BB, Ridge) = {rho_ridge:.3f}')
    print(f'  Spearman(BB, Tree-d1) = {rho_tree:.3f}')


=== GMSC: Feature Importance Comparison (Top 10) ===
                             Feature  Rank_BB  BB_SHAP  Rank_Ridge     Ridge  Rank_Tree   Tree_d1
RevolvingUtilizationOfUnsecuredLines        1 0.688012          10  0.066581          1 37.261452
NumberOfTime30-59DaysPastDueNotWorse        2 0.322286           1 26.449638          2 16.719545
                                 age        3 0.232390           2 25.046972          3 13.840671
             NumberOfTimes90DaysLate        4 0.212513           4  9.070575          5  9.463377
     NumberOfOpenCreditLinesAndLoans        5 0.162487           5  3.643845          4  9.482150
                           DebtRatio        6 0.122394           9  0.275400          8  4.847852
NumberOfTime60-89DaysPastDueNotWorse        7 0.111507           3 24.279064          7  5.475485
                       MonthlyIncome        8 0.095834           8  0.898718          6  6.027367
        NumberRealEstateLoansOrLines        9 0.059416          

## 5. Save Results

In [6]:
with open(OUT_DIR / 'results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print(f'Results saved to {OUT_DIR / "results.json"}')
print(f'\nOutput files:')
for p in sorted(OUT_DIR.iterdir()):
    s = p.stat().st_size
    label = f'{s/1024/1024:.1f}M' if s > 1e6 else f'{s/1024:.0f}K'
    print(f'  {p.name:45s} {label}')

Results saved to ..\outputs\NB03\results.json

Output files:
  contribs_EBM+mono_GMSC.npy                    1.1M
  contribs_EBM+mono_HC.npy                      9.6M
  contribs_EBM_GMSC.npy                         1.1M
  contribs_EBM_HC.npy                           9.6M
  contribs_OptBin+Ridge_GMSC.npy                1.1M
  contribs_OptBin+Ridge_HC.npy                  9.6M
  contribs_Ridge_GMSC.npy                       1.1M
  contribs_Ridge_HC.npy                         9.6M
  contribs_Tree-d1+mono_GMSC.npy                1.1M
  contribs_Tree-d1+mono_HC.npy                  9.6M
  contribs_Tree-d1_GMSC.npy                     1.1M
  contribs_Tree-d1_HC.npy                       9.6M
  contribs_Tree-d3_GMSC.npy                     1.1M
  contribs_Tree-d3_HC.npy                       9.6M
  contribs_Tree-d6_GMSC.npy                     1.1M
  contribs_Tree-d6_HC.npy                       9.6M
  results.json                                  5K


## 6. Scorecard Pipeline: Conversion + Pruning + Monotone

Tree-d1, Tree-d1+mono, EBM, EBM+mono → ScorecardModel로 변환하고, 변환 전후 성능을 비교한다.

- **Unpruned**: 원래 빈 구조 그대로
- **Pruned**: max_bins=5, min_ratio=5% (실무 배치 설정)

핵심 질문: 스코어카드 변환(이산화)이 R²와 AdvTop-k를 얼마나 감소시키는가?

In [7]:
SC_METHODS = {
    'Tree-d1':      lambda: TreeSurrogate(max_depth=1, verbose=0),
    'Tree-d1+mono': lambda: TreeSurrogate(max_depth=1, monotone_detect_mode='auto', verbose=0),
    'EBM':          lambda: EBMSurrogate(interactions=0, n_jobs=N_JOBS_EBM),
    'EBM+mono':     lambda: EBMSurrogate(interactions=0, monotone_detect_mode='auto', n_jobs=N_JOBS_EBM),
}

PRUNING = {
    'unpruned': dict(max_bins_per_feature=None, min_bin_ratio=0.0),
    'pruned':   dict(max_bins_per_feature=5, min_bin_ratio=0.05),
}

sc_results = {}

for data_name in datasets:
    d = datasets[data_name]
    X_train, X_test = d['X_train'], d['X_test']
    feature_names = d['feature_names']
    y_score_train = bb[data_name]['score_train']
    y_score_test = bb[data_name]['score_test']
    bb_shap_test = bb[data_name]['shap']
    bb_prob_test = bb[data_name]['prob']
    n_features = len(feature_names)
    y_binary_train = d['y_train']
    
    tv = bb[data_name]['tv_idx']
    X_tr = X_train.loc[tv['train_idx']]
    X_val = X_train.loc[tv['val_idx']]
    y_tr = y_score_train[X_train.index.get_indexer(tv['train_idx'])]
    y_val = y_score_train[X_train.index.get_indexer(tv['val_idx'])]
    
    print(f'\n{"="*80}')
    print(f'  {data_name}: Scorecard Pipeline')
    print(f'{"="*80}')
    
    sc_results[data_name] = {}
    
    for method_name, make_surr in SC_METHODS.items():
        t0 = time.time()
        surr = make_surr()
        
        if isinstance(surr, TreeSurrogate):
            surr.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        else:
            surr.fit(X=X_train, y_logit=y_score_train)
        
        # Raw surrogate baseline
        raw_pred = surr.predict(X_test)
        raw_contribs = surr.contributions(X_test)
        m_raw = evaluate(raw_contribs, raw_pred, bb_shap_test,
                        y_score_test, bb_prob_test, n_features)
        sc_results[data_name][f'{method_name}_raw'] = m_raw
        
        for prune_name, prune_params in PRUNING.items():
            sm = surr.to_scorecard_model(
                X_train, y_binary=np.array(y_binary_train),
                feature_names=feature_names, **prune_params
            )
            
            sc_pred = sm.predict(X_test)
            sc_contribs = sm.contributions(X_test)
            
            m_sc = evaluate(sc_contribs, sc_pred, bb_shap_test,
                           y_score_test, bb_prob_test, n_features)
            
            n_bins = sum(len(f.bins) for f in sm.features)
            n_active = len(sm.features)
            
            # Monotonicity compliance
            mono_count = 0
            for feat in sm.features:
                scores = [b.score for b in feat.bins]
                if len(scores) >= 2:
                    diffs = np.diff(scores)
                    if np.all(diffs >= -1e-8) or np.all(diffs <= 1e-8):
                        mono_count += 1
            mono_pct = round(mono_count / max(n_active, 1) * 100, 1)
            
            m_sc['n_bins'] = n_bins
            m_sc['n_active'] = n_active
            m_sc['mono_pct'] = mono_pct
            
            key = f'{method_name}_{prune_name}'
            sc_results[data_name][key] = m_sc
            
            delta_r2 = m_sc['R2'] - m_raw['R2']
            delta_at4 = m_sc['AdvTop4'] - m_raw['AdvTop4']
            print(f'  {key:25s}  R2={m_sc["R2"]:.3f}({delta_r2:+.3f})  '
                  f'AT4={m_sc["AdvTop4"]:.3f}({delta_at4:+.3f})  '
                  f'bins={n_bins}  feat={n_active}  mono={mono_pct}%')
        
        elapsed = time.time() - t0
        print(f'  {"(raw)":<25s}  R2={m_raw["R2"]:.3f}         '
              f'AT4={m_raw["AdvTop4"]:.3f}         ({elapsed:.0f}s)')
        print()


  GMSC: Scorecard Pipeline


  Tree-d1_unpruned           R2=0.936(+0.000)  AT4=0.930(+0.000)  bins=158  feat=10  mono=10.0%


  Tree-d1_pruned             R2=0.919(-0.017)  AT4=0.930(-0.001)  bins=38  feat=10  mono=70.0%
  (raw)                      R2=0.936         AT4=0.930         (10s)



  Tree-d1+mono_unpruned      R2=0.927(+0.000)  AT4=0.916(+0.000)  bins=134  feat=10  mono=100.0%


  Tree-d1+mono_pruned        R2=0.916(-0.011)  AT4=0.907(-0.009)  bins=36  feat=10  mono=100.0%
  (raw)                      R2=0.927         AT4=0.916         (10s)



...

  Tree-d1+mono_pruned        R2=0.834(-0.034)  AT4=0.605(-0.004)  bins=87  feat=29  mono=100.0%
  (raw)                      R2=0.869         AT4=0.609         (25s)



  EBM_unpruned               R2=0.871(-0.055)  AT4=0.787(-0.057)  bins=381  feat=41  mono=46.3%


  EBM_pruned                 R2=0.850(-0.076)  AT4=0.775(-0.069)  bins=136  feat=41  mono=75.6%
  (raw)                      R2=0.926         AT4=0.844         (425s)



  EBM+mono_unpruned          R2=0.811(-0.022)  AT4=0.542(-0.004)  bins=533  feat=41  mono=95.1%


  EBM+mono_pruned            R2=0.789(-0.045)  AT4=0.542(-0.004)  bins=135  feat=41  mono=100.0%
  (raw)                      R2=0.834         AT4=0.546         (95s)



In [8]:
# Save Section 6 results alongside Section 2 results
with open(OUT_DIR / 'scorecard_results.json', 'w') as f:
    json.dump(sc_results, f, indent=2)
print(f'Scorecard results saved to {OUT_DIR / "scorecard_results.json"}')

Scorecard results saved to ..\outputs\NB03\scorecard_results.json
